## Check which kernel is running

In [ ]:
import sys
print(sys.executable)
print(sys.version)

## Importing the libraries and the dataset

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("..") / "data" / "raw" / "data-zuerich_2026-10-29.csv"
print(DATA.resolve(), DATA.exists())   
df = pd.read_csv(DATA, encoding="utf-8", low_memory=False)

## Print size of the table

In [ ]:
print(df.shape)

## print data period

In [ ]:
print(pd.to_datetime(df["requested_datetime"]).agg(["min", "max"]))

## Portion without text

In [ ]:
print(df["detail"].isna().mean())   

## Count the number of words per message

In [ ]:
df["woerter"] = df["detail"].fillna("").str.split().str.len()

## Statistics on text lengths 
## (count, mean, std, min, max, quartiles)

In [ ]:
print(df["woerter"].describe())    

## Percentage with encoding errors

In [ ]:
print(df["detail"].str.contains("Ã|Â", na=False).mean()) 

## Most common categories (first 10 entries)

In [ ]:
print(df["service_code"].value_counts().head(10))

## Setup for the following checks
Project config (seed) and `ftfy` to repair the encoding of texts that are read manually.

In [ ]:
sys.path.insert(0, str(Path("..").resolve()))
from src.config import load_config
import ftfy

cfg = load_config()
SEED = cfg["seed"]

## B1 Proxy sample: defects vs. complaints
50 reports are drawn reproducibly and labelled manually in `data/sample/proxy_sample.csv`
(open in VS Code, not Excel). Labels are defined before reading the reports:

| Label | Criterion | Example |
|---|---|---|
| `defect` | reports a condition that should be fixed | "Lampe defekt" |
| `complaint` | criticises action or inaction of the administration or others, lack of enforcement | "Halteverbot wird nicht durchgesetzt" |
| `other` | question, suggestion, praise, unclear | "Könnte hier eine Bank hin?" |

Borderline rule: a report that criticises inaction ("schon zum dritten Mal gemeldet") counts as `complaint`.

In [ ]:
SAMPLE_FILE = Path("..") / "data" / "sample" / "proxy_sample.csv"
SAMPLE_SIZE = 50
LABELS = ["defect", "complaint", "other"]

if SAMPLE_FILE.exists():
    # never overwrite: the file contains the manual labels
    proxy = pd.read_csv(SAMPLE_FILE, encoding="utf-8", keep_default_na=False)
    print(f"Loaded existing sample from {SAMPLE_FILE}")
else:
    proxy = (
        df[df["detail"].notna()]
        .sample(n=SAMPLE_SIZE, random_state=SEED)
        [["service_request_id", "requested_datetime", "service_code", "detail"]]
        .assign(
            service_code=lambda d: d["service_code"].map(ftfy.fix_text),
            detail=lambda d: d["detail"].map(ftfy.fix_text),
            label="",
        )
    )
    proxy.to_csv(SAMPLE_FILE, index=False, encoding="utf-8")
    print(f"Saved new sample to {SAMPLE_FILE}: fill in the 'label' column manually")
proxy.head()

In [ ]:
Z_95 = 1.96  # z-value for a 95 % confidence level


def wilson_interval(successes, n, z=Z_95):
    """Wilson score interval for a proportion (more reliable than p ± z·SE for small n)."""
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half_width = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / denom
    return centre - half_width, centre + half_width


labels = proxy["label"].str.strip()
missing = (labels == "").sum()
if missing:
    print(f"{missing} of {len(proxy)} reports are not labelled yet.")
else:
    unknown = set(labels) - set(LABELS)
    assert not unknown, f"unknown labels: {unknown}"
    counts = labels.value_counts().reindex(LABELS, fill_value=0)
    for label, k in counts.items():
        low, high = wilson_interval(k, len(labels))
        print(f"{label:<10}{k:>4}  {k / len(labels):6.1%}   95 % CI {low:.0%} to {high:.0%}")

## B2 Distribution of `userid`
Is one id dominant (e.g. a collective id for anonymous or app reports)? If so, no statements about frequent reporters are possible.

In [ ]:
user_share = df["userid"].value_counts(normalize=True)
print("distinct user ids:", df["userid"].nunique())
user_share.head(10)

In [ ]:
top_user = user_share.index[0]
is_top_user = df["userid"] == top_user
print(f"most frequent id {top_user}: {is_top_user.mean():.1%} of all reports")

year = pd.to_datetime(df["requested_datetime"]).dt.year
print()
print("share per year:")
print(is_top_user.groupby(year).mean().round(3).to_string())
print()
print("share per channel:")
print(is_top_user.groupby(df["interface_used"]).agg(["mean", "size"]).round(3).to_string())

**Interpretation:** *(to be written)*

## B3 Short texts

In [ ]:
SHORT_MAX_WORDS = 3
is_short = df["woerter"] <= SHORT_MAX_WORDS
print(f"reports with <= {SHORT_MAX_WORDS} words: {is_short.sum()} ({is_short.mean():.1%})")
df.loc[is_short, "detail"].fillna("").map(ftfy.fix_text).value_counts().head(15)

### Decision: all texts are used
Short reports such as "Abfall" or "Schrottvelo" are thematically clear. Excluding them would distort the topic frequencies.

Consequences to discuss with the results:
- Identical short texts get identical vectors. Repeated reports like "Abfall" form a dense point that KMeans is likely to turn into a cluster of its own.
- One-word texts add little word context to NMF, which can lower topic coherence.
- Optional robustness check: compare topic shares of short and longer reports.